In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import os
import time
import numpy as np
from itertools import chain
import pickle
from glob import glob
from pathlib import Path
from copy import deepcopy

from multiprocessing import Pool

from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from utils.write_output import write_array2txt
from utils.math_methods import project_point_onto_plane

from cylinder_extraction.utils import detect_objects

# Read point cloud

In [2]:
data_dir = Path('D:/REGISTRATION DATA/External/JB-483 70ft EXT')
filename_template = '{}.e57'
ptc_col_names =  ["x","y","z"]


In [ ]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn[-3:]
    station_names.append(sn)
    ptc_fn = filename_template.format(sn)
    print(ptc_fn)
    station = Station(
        name=sn, file_path=fp
    )
    station.extract_cylinder(fit=True, refine=True)
    station.extract_plane()
    station.create_artificial_cylinders()
    # create a directory for saving outputs
    out_dir = os.path.join(data_dir,station.name + '-output_test')
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    # save station without points
    station.ptc = None
    out_fp = os.path.join(out_dir, f'{station.name}.pkl')
    with open(out_fp, 'wb') as f:
        pickle.dump(station, f)

In [3]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
stations = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn[-3:]
    station_names.append(sn)
    ptc_fn = filename_template.format(sn)
    print(ptc_fn)
    station = Station(
        name=sn, file_path=fp
    )
    stations.append(station)

with Pool(12) as p:
    stations = p.map(
        detect_objects,
        stations
    )

STATION Jb483 Scan_SuncorTk20D71Scan_055
055.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_055.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_056
056.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_056.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_057
057.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_057.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_058
058.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_058.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_059
059.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_059.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_060
060.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483 Scan_SuncorTk20D71Scan_060.e57
[0 0 0 ... 0 0 0]
STATION Jb483 Scan_SuncorTk20D71Scan_061
061.e57
D:\REGISTRATION DATA\External\JB-483 70ft EXT Jb483

In [4]:
for station in stations:
    # write clusters
    for cltr in station.extended_clusters:
        write_array2txt(
            cltr['filepath'],
            cltr['points']
        )
    station.extended_clusters = None
    # write station metadata
    out_dir = os.path.join(data_dir,station.name + '-output')
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    # save station without points
    station.ptc = None
    out_fp = os.path.join(out_dir, f'{station.name}.pkl')
    with open(out_fp, 'wb') as f:
        pickle.dump(station, f)

In [ ]:
source_files = glob(str(data_dir / '*.e57'))
station_names = []
for fp in source_files:
    if 'roof' in fp.lower():
        continue
    fn = Path(fp).stem
    print(f'STATION {fn}')
    # station id
    sn = fn.split('_')[-1]
    station_names.append(sn)
    ptc_fn = filename_template.format(sn)
    print(ptc_fn)
    station = Station(
        name=sn, file_path=fp
    )
    # station.write_points()
    station.extract_cylinder(refine=True)
    out_dir = os.path.join(data_dir,station.name + '-output')
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    # save station without points
    station.ptc = None
    out_fp = os.path.join(out_dir, f'{station.name}.pkl')
    with open(out_fp, 'wb') as f:
        pickle.dump(station, f)
# # data/JB-483/Jb-483 Scan_SuncorTk20D71Scan_055.e57
# for sn in station_names:
#     ptc_fn = filename_template.format(sn)
#     print(ptc_fn)
#     station = Station(
#         name=sn, file_path=os.path.join(data_dir, ptc_fn)
#     )
#     # station.write_points()
#     station.extract_cylinder()
#     out_dir = os.path.join(data_dir,station.name + '-output')
#     if not os.path.exists(out_dir):
#         os.mkdir(out_dir)
#     out_fp = os.path.join(out_dir, f'{station.name}.pkl')
#     with open(out_fp, 'wb') as f:
#         pickle.dump(station, f)

In [5]:
stations[-1].planes

[{'id': 79,
  'original_cluster': 1,
  'filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\067-output\\067 cluster_79.txt',
  'points': array([[-14.88313596,   6.43612595, 558.88055643],
         [-14.8837243 ,   6.43653512, 559.68630752],
         [-14.88188518,   6.43621723, 559.65383987],
         ...,
         [  8.97260056,  19.69791598, 555.43652808],
         [  8.97319458,  19.69744389, 555.41499472],
         [  8.97389691,  19.69688844, 555.50100239]]),
  'equations': [-0.48503018541635257,
   0.874422005446211,
   0.011483711351540887,
   -19.29249357780617]},
 {'id': 79,
  'original_cluster': 9,
  'filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\067-output\\067 cluster_79.txt',
  'points': array([[ 29.66215611, -41.84442918, 558.17749452],
         [ 29.20524191, -42.12383057, 557.78322802],
         [ 29.65960136, -41.86046526, 557.77930533],
         ...,
         [ 37.56589707, -37.49960915, 558.85343297],
         [ 37.56556302, -37.49131755, 55

In [10]:
np.dot(
    np.array([stations[0].planes[0]['equations'][:3]]).ravel(),
    np.array([stations[0].planes[1]['equations'][:3]]).ravel()
)

-0.0024700400445711907

In [9]:
np.array([stations[0].planes[0]['equations'][:3]]).ravel()

array([ 0.37871817, -0.92544622, -0.01103827])

In [11]:
st55 = stations[0]
st56 = stations[1]

In [18]:
st56.refined_cylinders

[{'station': {'name': '056'}, 'radius': 3145.3115, 'axis': "{'position': array([  404.21902 , -3081.945   ,   -23.742477], dtype=float32), 'vector': array([ 0.0010669 , -0.00756514,  0.9999708 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\056-output\\056 cluster_9.txt'},
 {'station': {'name': '056'}, 'radius': 10.697558, 'axis': "{'position': array([18.89641   , -2.4692802 ,  0.01995429], dtype=float32), 'vector': array([-1.067589e-03, -7.892327e-05,  9.999994e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\056-output\\056 cluster_11.txt'},
 {'station': {'name': '056'}, 'radius': 0.001, 'axis': "{'position': array([13.31819287, 41.81101727,  0.57480789]), 'vector': array([0, 0, 1])}", 'ptc_filepath': None}]

In [16]:
project_point_onto_plane(
    st55.refined_cylinders[0].axis.intersect_xy(),
    st55.planes[0]['equations']
)

(array([13.53955799, 24.68304364,  0.47411681]), 42.95207671990831)

In [19]:
project_point_onto_plane(
    st56.refined_cylinders[1].axis.intersect_xy(),
    st56.planes[0]['equations']
)

(array([13.31819287, 41.81101727,  0.57480789]), 44.63397522668208)

In [21]:
st56.refined_cylinders

[{'station': {'name': '056'}, 'radius': 3145.3115, 'axis': "{'position': array([  404.21902 , -3081.945   ,   -23.742477], dtype=float32), 'vector': array([ 0.0010669 , -0.00756514,  0.9999708 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\056-output\\056 cluster_9.txt'},
 {'station': {'name': '056'}, 'radius': 10.697558, 'axis': "{'position': array([18.89641   , -2.4692802 ,  0.01995429], dtype=float32), 'vector': array([-1.067589e-03, -7.892327e-05,  9.999994e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\JB-483 70ft EXT\\056-output\\056 cluster_11.txt'},
 {'station': {'name': '056'}, 'radius': 0.001, 'axis': "{'position': array([13.31819287, 41.81101727,  0.57480789]), 'vector': array([0, 0, 1])}", 'ptc_filepath': None}]